# DFT XC skeleton 二阶导数分解 (B3LYP, GGA)

对标 `06-2-decomp_de_xc_tpss0.ipynb`，但 B3LYP 是 hybrid GGA，无 tau 贡献。我们仍采用与 TPSS0 一致的策略：

- **fxc 部分**：直接使用 rho 导数（每原子方向给出 `[4, ngrids]` 的一阶密度导数），与 fxc 核 `[4, 4, ngrids]` 直接缩并。
- **vxc 部分**（non-diagonal）：先给出原子非依赖的 `[3, 3, nao, nao]` 二阶分量矩阵，再依双原子加权求和。
- **vxc 部分**（diagonal）：通过 ao 三阶导数与 ao_dm0 缩并。

相比 TPSS0，这里不再有 tau (deriv=4) 相关项。


In [1]:
from pyscf import gto, dft, lib
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
import sys
sys.path.append("..")

from pyhessref.nimatmul.becke_partition import becke_partition
from pyhessref.nimatmul import rks as rks_nimatmul

In [3]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [4]:
mf = dft.RKS(mol, xc="B3LYP").density_fit()
dat0 = np.load("nh3_r_b3lyp.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True

In [5]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mocc = mo_coeff[:, mo_occ > 0]
dm0 = mocc @ mocc.T * 2
natm = mol.natm
nao = mol.nao
aoslices = mol.aoslice_by_atom()
ni = dft.numint.NumInt()

In [6]:
# grids = dft.grid.Grids(mol)
# grids.coords = coords = dat0["grid_coords"]
# grids.weights = weights = dat0["grid_weights"]
# ngrids = len(weights)

In [7]:
grids = dft.gen_grid.Grids(mol)
grids.build(sort_grids=False)
coords = grids.coords
weights = grids.weights
ngrids = len(weights)

In [8]:
# Reference de_vxc from 06-4: this is what we want to reproduce.
de_ks_ref = np.load("nh3_r_b3lyp_decomp.npz")["de_vxc"]
print("de_vxc_ref shape:", de_ks_ref.shape)
print("de_vxc_ref fp:   ", lib.fp(de_ks_ref))

de_vxc_ref shape: (4, 4, 3, 3)
de_vxc_ref fp:    -0.8985828139605962


In [9]:
ao = ni.eval_ao(mol, grids.coords, deriv=3)
rho = ni.eval_rho2(mol, ao[:4], mo_coeff, mo_occ, xctype="GGA")
print("rho shape:", rho.shape)

rho shape: (4, 43328)


In [10]:
xc_eff = ni.eval_xc_eff(mf.xc, rho, deriv=2, xctype="GGA")
vxc = xc_eff[1]  # shape [4, ngrid]
fxc = xc_eff[2]  # shape [4, 4, ngrid]
print("vxc shape:", vxc.shape, "fxc shape:", fxc.shape)

vxc shape: (4, 43328) fxc shape: (4, 4, 43328)


In [11]:
TX, TY, TZ = 0, 1, 2
O = 0
X, Y, Z = 1, 2, 3
XX, XY, XZ = 4, 5, 6
YX, YY, YZ = 5, 7, 8
ZX, ZY, ZZ = 6, 8, 9
XXX, XXY, XXZ, XYY, XYZ, XZZ = 10, 11, 12, 13, 14, 15
YYY, YYZ, YZZ, ZZZ = 16, 17, 18, 19

In [12]:
ao_dm0 = ao @ dm0
ao_dm0.shape

(20, 43328, 49)

### fxc contribution

In [13]:
# B3LYP (GGA): no tau, drho has 4 components (RHO, GRAD_X, GRAD_Y, GRAD_Z)
drho = np.zeros((natm, 3, 4, ngrids))
for A in range(natm):
    _, _, p0, p1 = aoslices[A]
    slc = slice(p0, p1)
    ao_slc = ao[:, :, slc]
    ao_dm0_slc = ao_dm0[:, :, slc]
    DERIV_COMPONENTS = [
        # RHO part
        [(TX, 0), (X, O)],
        [(TY, 0), (Y, O)],
        [(TZ, 0), (Z, O)],
        # SIGMA part (bra deriv 2)
        [(TX, X), (XX, O)],
        [(TX, Y), (XY, O)],
        [(TX, Z), (XZ, O)],
        [(TY, X), (YX, O)],
        [(TY, Y), (YY, O)],
        [(TY, Z), (YZ, O)],
        [(TZ, X), (ZX, O)],
        [(TZ, Y), (ZY, O)],
        [(TZ, Z), (ZZ, O)],
        # SIGMA part (bra deriv 1, ket deriv 1)
        [(TX, X), (X, X)],
        [(TX, Y), (X, Y)],
        [(TX, Z), (X, Z)],
        [(TY, X), (Y, X)],
        [(TY, Y), (Y, Y)],
        [(TY, Z), (Y, Z)],
        [(TZ, X), (Z, X)],
        [(TZ, Y), (Z, Y)],
        [(TZ, Z), (Z, Z)],
    ]
    for ((t, v), (cbra, cket)) in DERIV_COMPONENTS:
        drho[A, t, v] -= np.einsum("gu, gu -> g", ao_slc[cbra], ao_dm0_slc[cket])
# scale symmetric coeff: all 4 components (RHO + SIGMA) get *2
drho *= 2

In [14]:
lib.fp(drho)

np.float64(-3951374.647722333)

In [15]:
de_fxc = np.einsum("g, Atxg, xyg, Bsyg -> ABts", weights, drho, fxc, drho)
print(lib.fp(de_fxc))

-21.24987446516278


### dao_vxc_diag contribution

In [16]:
# --- dao_vxc_diag (GGA: no tau) --- #
dao_vxc_diag = np.zeros((6, nao))  # 6 denotes xx, xy, xz, yy, yz, zz
wv = weights * vxc  # [4, ngrids]

# Contribution 1: ao[i+4]^T @ (wv[0]*ao[0] + wv[1]*ao[1] + wv[2]*ao[2] + wv[3]*ao[3])
aow_diag = (np.einsum("gu, g -> gu", ao_dm0[0], wv[0])
          + np.einsum("gu, g -> gu", ao_dm0[1], wv[1])
          + np.einsum("gu, g -> gu", ao_dm0[2], wv[2])
          + np.einsum("gu, g -> gu", ao_dm0[3], wv[3]))
for idx, its in enumerate([XX, XY, XZ, YY, YZ, ZZ]):
    dao_vxc_diag[idx] += 2 * np.einsum("gu, gu -> u", ao[its], aow_diag)

# Contribution 2 (GGA triple-derivative part)
TRIPLE_DERIV_DIAG = [
    [XXX, XXY, XXZ],  # xx
    [XXY, XYY, XYZ],  # xy
    [XXZ, XYZ, XZZ],  # xz
    [XYY, YYY, YYZ],  # yy
    [XYZ, YYZ, YZZ],  # yz
    [XZZ, YZZ, ZZZ],  # zz
]
for idx, (i3x, i3y, i3z) in enumerate(TRIPLE_DERIV_DIAG):
    aow_triple = (np.einsum("gu, g -> gu", ao[i3x], wv[1])
                + np.einsum("gu, g -> gu", ao[i3y], wv[2])
                + np.einsum("gu, g -> gu", ao[i3z], wv[3]))
    dao_vxc_diag[idx] += 2 * np.einsum("gu, gu -> u", aow_triple, ao_dm0[0])

de_vxc_diag = np.zeros((natm, natm, 6))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    de_vxc_diag[A, A] += np.einsum("Au -> A", dao_vxc_diag[:, slcA])
de_vxc_diag = de_vxc_diag[:, :, [[0, 1, 2], [1, 3, 4], [2, 4, 5]]]
print("de_vxc_diag fp:", lib.fp(de_vxc_diag))

de_vxc_diag fp: 49.68876638573063


### dao_vxc_off contribution

In [17]:
# --- dao_vxc (GGA: no tau) --- #
wv = weights * vxc  # [4, ngrids]
dao_vxc = np.zeros((3, 3, nao, nao))

# GGA part (RHO + SIGMA)
GGA_CALLS = [[XX, XY, XZ], [YX, YY, YZ], [ZX, ZY, ZZ]]

aowv = [None, None, None]
for t in range(3):
    aowv[t] = 0.5 * np.einsum("gu, g -> gu", ao[t + 1], wv[0])
    for r in range(3):
        aowv[t] += np.einsum("gu, g -> gu", ao[GGA_CALLS[t][r]], wv[r + 1])

for t in range(3):
    for s in range(3):
        dao_vxc[t, s] += 2 * aowv[s].T @ ao[t + 1]     # ipip[t,s]

dao_vxc += dao_vxc.transpose(1, 0, 3, 2)  # [s,t] with AO indices transposed

de_vxc_off = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    for B in range(A + 1):
        _, _, p0B, p1B = aoslices[B]
        slcB = slice(p0B, p1B)
        de_vxc_off[A, B] += np.einsum("tsuv, uv -> ts", dao_vxc[:, :, slcB, slcA], dm0[slcB, slcA])
        if A != B:
            de_vxc_off[B, A] = de_vxc_off[A, B].T
print("de_vxc fp:", lib.fp(de_vxc_off))

de_vxc fp: -29.337474734527564


### summarize of common DFT contribution

In [18]:
de_xc_recap = de_vxc_diag + de_vxc_off + de_fxc
assert np.allclose(de_xc_recap, de_ks_ref)

In [19]:
dat = dict(np.load("nh3_r_b3lyp_decomp.npz"))
dat.update({
    "de_vxc_diag": de_vxc_diag,
    "de_vxc_off": de_vxc_off,
    "de_fxc": de_fxc,
})
np.savez("nh3_r_b3lyp_decomp.npz", **dat)

In [20]:
de_xc_recap.sum(axis=(0, 1))

array([[ 0.0004 , -0.00001,  0.     ],
       [-0.00001,  0.00034,  0.00003],
       [ 0.     ,  0.00003,  0.00037]])

In [21]:
np.abs(de_xc_recap.sum(axis=(0, 1))).max()

np.float64(0.0003977856045045136)

### becke partition derivative (preparation)

In [22]:
ngrids = grids.weights.size
ni = dft.numint.NumInt()
ao = ni.eval_ao(mol, grids.coords, deriv=2)
dm0 = mf.make_rdm1()
ao_dm0 = ao @ dm0
rho, exc, vxc, fxc = rks_nimatmul._eval_rho_exc_vxc_fxc("B3LYP", "GGA", ao, ao_dm0)
drho = rks_nimatmul._make_drho("GGA", ao, ao_dm0, mol.aoslice_by_atom())

In [23]:
natm = mol.natm
becke_scheme = grids.radii_adjust(mol, grids.atomic_radii)
adjustment_factor = np.array([becke_scheme(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)
becke_result = becke_partition(grids.coords, mol.atom_coords(), grids.atm_idx, grids.quadrature_weights, adjustment_factor, 3, 512, 2, None)
w, dw, ddw = becke_result["w"], becke_result["dw"], becke_result["ddw"]

### t1

In [89]:
t1 = np.einsum("Atg, xg, Bsxg -> ABts", dw, vxc, drho)
t1 += np.einsum("ABts -> BAst", t1)

In [121]:
t1[0, 1]

array([[-0.93322, -0.05716, -0.16004],
       [-0.08997,  0.0516 , -0.01137],
       [-0.18441, -0.0075 ,  0.01248]])

### t2

In [90]:
t2 = np.einsum("AtBsg, g, g -> ABts", ddw, exc, rho[0])

In [122]:
t2[0, 1]

array([[ 0.001  , -0.1181 , -0.13659],
       [ 0.00622,  0.30383, -0.00699],
       [-0.0326 , -0.02132,  0.3082 ]])

### t3

In [114]:
t3 = np.zeros((natm, natm, 3, 3))

dsum_rho = drho.sum(axis=0)
for A in range(natm):
    maskA = grids.atm_idx == A
    t3[A] -= np.einsum("g, txg, xyg, Bsyg -> Bts", weights[maskA], dsum_rho[..., maskA], fxc[..., maskA], drho[..., maskA])
t3 += np.einsum("ABts -> BAst", t3)

In [115]:
t3[0, 1]

array([[-0.02617, -0.04418, -0.05123],
       [ 0.01006,  0.08572, -0.00139],
       [-0.00921, -0.00793,  0.08664]])

### t4

In [99]:
# --- dao_vxc (GGA: no tau) --- #
wv = weights * vxc  # [4, ngrids]
dao_vxc_grids = np.zeros((natm, 3, 3, nao, nao))

# GGA part (RHO + SIGMA)
GGA_CALLS = [[XX, XY, XZ], [YX, YY, YZ], [ZX, ZY, ZZ]]

aowv = [None, None, None]
for t in range(3):
    aowv[t] = 0.5 * np.einsum("gu, g -> gu", ao[t + 1], wv[0])
    for r in range(3):
        aowv[t] += np.einsum("gu, g -> gu", ao[GGA_CALLS[t][r]], wv[r + 1])

for A in range(natm):
    maskA = grids.atm_idx == A
    for t in range(3):
        for s in range(3):
            dao_vxc_grids[A, t, s] += 2 * aowv[s][maskA].T @ ao[t + 1][maskA]     # ipip[t,s]

dao_vxc_grids += dao_vxc_grids.transpose(0, 2, 1, 4, 3)  # [s,t] with AO indices transposed

In [123]:
t4 = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    for B in range(natm):
        _, _, p0B, p1B = aoslices[B]
        slcB = slice(p0B, p1B)
        t4[A, B] += np.einsum("tsuv, uv -> ts", dao_vxc_grids[A][:, :, slcB], dm0[slcB])
t4 += np.einsum("ABts -> BAst", t4)

In [124]:
t4[0, 1]

array([[-0.57059,  0.00422, -0.03114],
       [ 0.02563, -0.2894 ,  0.01017],
       [-0.01353,  0.0075 , -0.29313]])

### t5

In [125]:
t5 = np.zeros((natm, natm, 3, 3))

dsum_rho = drho.sum(axis=0)
for A in range(natm):
    maskA = grids.atm_idx == A
    t5[A] -= np.einsum("Bsg, xg, txg -> Bts", dw[..., maskA], vxc[..., maskA], dsum_rho[..., maskA])
t5 += np.einsum("ABts -> BAst", t5)

In [127]:
t5[0, 1]

array([[-0.00114,  0.11828,  0.13659],
       [-0.00597, -0.30408,  0.00691],
       [ 0.0325 ,  0.02132, -0.3081 ]])

In [139]:
(t2 + t5)[0, 1]

array([[-0.00014,  0.00017, -0.00001],
       [ 0.00024, -0.00026, -0.00009],
       [-0.0001 , -0.     ,  0.00009]])

### t6

In [133]:
t1[0, 1]

array([[-0.93322, -0.05716, -0.16004],
       [-0.08997,  0.0516 , -0.01137],
       [-0.18441, -0.0075 ,  0.01248]])

In [134]:
t3[0, 1]

array([[-0.02617, -0.04418, -0.05123],
       [ 0.01006,  0.08572, -0.00139],
       [-0.00921, -0.00793,  0.08664]])

In [135]:
t4[0, 1]

array([[-0.57059,  0.00422, -0.03114],
       [ 0.02563, -0.2894 ,  0.01017],
       [-0.01353,  0.0075 , -0.29313]])

In [132]:
(t3 + t4 + t1)[0, 1]

array([[-1.52998, -0.09712, -0.24241],
       [-0.05427, -0.15209, -0.0026 ],
       [-0.20714, -0.00792, -0.19401]])

In [119]:
t1[0, 1]

array([[-0.93322, -0.05716, -0.16004],
       [-0.08997,  0.0516 , -0.01137],
       [-0.18441, -0.0075 ,  0.01248]])

In [120]:
t2[0, 1]

array([[ 0.001  , -0.1181 , -0.13659],
       [ 0.00622,  0.30383, -0.00699],
       [-0.0326 , -0.02132,  0.3082 ]])